# This notebook creates visualisations for the data

In [ ]:
import torch
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

# Force notebook CWD to project root: .../pde_lightning
PROJECT_ROOT = Path.cwd().resolve().parent  # from notebooks/ -> parent
os.chdir(PROJECT_ROOT)

# Keep imports stable
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())

print("PROJECT_ROOT:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

In [ ]:
def load_data(data_folder):
    data_train = torch.load(os.path.join(data_folder, "data_train.pt"))
    data_test_id = torch.load(os.path.join(data_folder, "data_test_id.pt"))
    data_test_od = torch.load(os.path.join(data_folder, "data_test_od.pt"))
    return data_train, data_test_id, data_test_od

def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.cpu().numpy()
    else:
        return np.asarray(x)

def create_figure(
    data_id,
    data_od,
    id_index=[0, 6, 1],
    od_index=[0, 3, 2],
    parameter_name=("c",),   # accepts "c" or ["c"] or ["beta", "nu"]
    fontsize=10,
):
    plt.close("all")

    # Normalize parameter_name to a list
    if isinstance(parameter_name, str):
        parameter_names = [parameter_name]
    else:
        parameter_names = list(parameter_name)

    def format_param_title(param_row):
        vals = np.atleast_1d(to_numpy(param_row)).astype(float).ravel()
        n = min(len(parameter_names), len(vals))
        return " ".join([f"{parameter_names[i]}={vals[i]:.3f}" for i in range(n)])

    fig = plt.figure(figsize=(10, 3.2))
    gs_1 = fig.add_gridspec(
        ncols=3, nrows=2, bottom=0.1, left=0.05, top=0.85, right=0.46, wspace=0.4, hspace=0.3
    )
    gs_2 = fig.add_gridspec(
        ncols=3, nrows=2, bottom=0.1, left=0.54, top=0.85, right=0.95, wspace=0.4, hspace=0.3
    )

    t = to_numpy(data_id["t"]).squeeze()
    x = to_numpy(data_id["x"]).squeeze()
    params_id = to_numpy(data_id["parameter"])
    params_od = to_numpy(data_od["parameter"])

    for i, b in enumerate(id_index[:3]):
        ax = fig.add_subplot(gs_1[0, i])
        z = to_numpy(data_id["states"][b, :, :, 0]).T
        im = ax.imshow(z, origin="lower", aspect="auto", extent=[t.min(), t.max(), x.min(), x.max()])

        ax.set_title(format_param_title(params_id[b]), fontsize=fontsize)
        ax.set_xlabel("t", fontsize=fontsize)

        if i == 0:
            ax.set_ylabel("x", fontsize=fontsize)
            ax.tick_params(axis="y", labelsize=fontsize)
        else:
            ax.set_ylabel("")
            ax.tick_params(axis="y", left=False, labelleft=False)

        ax.tick_params(axis="x", labelsize=fontsize)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=fontsize)

    for i, b in enumerate(od_index[:3]):
        ax = fig.add_subplot(gs_2[0, i])
        z = to_numpy(data_od["states"][b, :, :, 0]).T
        im = ax.imshow(z, origin="lower", aspect="auto", extent=[t.min(), t.max(), x.min(), x.max()])

        ax.set_title(format_param_title(params_od[b]), fontsize=fontsize)
        ax.set_xlabel("t", fontsize=fontsize)
        ax.set_ylabel("")
        ax.tick_params(axis="y", left=False, labelleft=False)
        ax.tick_params(axis="x", labelsize=fontsize)

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=fontsize)

    fig.text(0.26, 0.98, "In-domain", ha="center", fontsize=fontsize, transform=fig.transFigure)
    fig.text(0.74, 0.98, "Out-of-domain", ha="center", fontsize=fontsize, transform=fig.transFigure)
    return fig


In [ ]:
equation = "Advection"
data_folder = f"data/processed/{equation}"
data_train, data_test_id, data_test_od = load_data(data_folder)

fig = create_figure(data_test_id, data_test_od, id_index=[0, 6, 1], od_index=[0, 3, 2], fontsize=9, parameter_name =[r"$\beta$"])
fig.savefig(f"outputs/data_visualisation_{equation}.pdf", bbox_inches="tight")

In [ ]:
equation = "Burgers"
data_folder = f"data/processed/{equation}"
data_train, data_test_id, data_test_od = load_data(data_folder)

fig = create_figure(data_test_id, data_test_od, id_index=[1, 0, 7], od_index=[3,10, 2], fontsize=9, parameter_name =[r"$\nu$"])
fig.savefig(f"outputs/data_visualisation_{equation}.pdf", bbox_inches="tight")

In [ ]:
equation = "ReactionDiffusion_1D"
data_folder = f"data/processed/{equation}"
data_train, data_test_id, data_test_od = load_data(data_folder)

fig = create_figure(data_test_id, data_test_od, id_index=[2, 0, 7], od_index=[1,3, 5], fontsize=9, parameter_name =[r"$\rho$", r"$\nu$"])
fig.savefig(f"outputs/data_visualisation_{equation}.pdf", bbox_inches="tight")

In [ ]:
# Visualisations of 2D data

def create_figure_2d(
    data_id,
    data_od,
    id_index=[0, 1, 2],
    od_index=[0, 1, 2],
    parameter_name=("c",),
    fontsize=10,
):
    plt.close("all")

    # Normalize parameter_name to a list
    if isinstance(parameter_name, str):
        parameter_names = [parameter_name]
    else:
        parameter_names = list(parameter_name)

    def format_param_title(param_row):
        vals = np.atleast_1d(to_numpy(param_row)).astype(float).ravel()
        n = min(len(parameter_names), len(vals))
        return " ".join([f"{parameter_names[i]}={vals[i]:.3f}" for i in range(n)])

    # Columns correspond to normalized time positions 0, 0.5, and 1.
    time_positions = [0.0, 0.5, 1.0]

    t = to_numpy(data_id["t"]).squeeze().astype(float)
    x = to_numpy(data_id["x"]).squeeze().astype(float)
    y = to_numpy(data_id["y"]).squeeze().astype(float) if "y" in data_id else x

    t_min, t_max = float(np.min(t)), float(np.max(t))
    t_targets = np.array([t_min + p * (t_max - t_min) for p in time_positions])
    t_idx = [int(np.argmin(np.abs(t - tt))) for tt in t_targets]
    t_targets = t_targets-1 

    params_id = to_numpy(data_id["parameter"])
    params_od = to_numpy(data_od["parameter"])

    fig = plt.figure(figsize=(10, 5.5))
    gs_1 = fig.add_gridspec(
        ncols=3, nrows=3, bottom=0.08, left=0.05, top=0.9, right=0.44, wspace=0.13, hspace=0.35
    )
    gs_2 = fig.add_gridspec(
        ncols=3, nrows=3, bottom=0.08, left=0.56, top=0.9, right=0.95, wspace=0.13, hspace=0.35
    )

    # Keep axes grouped by row so each row can get an independent colorbar and limits.
    row_axes_id = [[] for _ in range(3)]
    row_axes_od = [[] for _ in range(3)]

    for r, b in enumerate(id_index[:3]):
        row_slices = [to_numpy(data_id["states"][b, idx, :, :, 0]).T for idx in t_idx]
        row_vmin = min(np.min(z) for z in row_slices)
        row_vmax = max(np.max(z) for z in row_slices)

        last_im = None
        for c, z in enumerate(row_slices):
            ax = fig.add_subplot(gs_1[r, c])
            row_axes_id[r].append(ax)
            last_im = ax.imshow(
                z,
                origin="lower",
                aspect="auto",
                extent=[x.min(), x.max(), y.min(), y.max()],
                vmin=row_vmin,
                vmax=row_vmax,
            )

            if r == 0:
                ax.set_title(f"T={t_targets[c]:.2f}", fontsize=fontsize)
            if r == 2:
                ax.set_xlabel("x", fontsize=fontsize)
            else:
                ax.tick_params(axis="x", labelbottom=False)

            if c == 0:
                ax.set_ylabel("y", fontsize=fontsize)
                ax.text(
                    -0.55,
                    0.5,
                    format_param_title(params_id[b]),
                    transform=ax.transAxes,
                    rotation=90,
                    va="center",
                    ha="right",
                    fontsize=fontsize,
                )
            else:
                ax.tick_params(axis="y", labelleft=False)

            ax.tick_params(axis="both", labelsize=fontsize)

        cbar = fig.colorbar(last_im, ax=row_axes_id[r], fraction=0.02, pad=0.01)
        cbar.ax.tick_params(labelsize=fontsize)

    for r, b in enumerate(od_index[:3]):
        row_slices = [to_numpy(data_od["states"][b, idx, :, :, 0]).T for idx in t_idx]
        row_vmin = min(np.min(z) for z in row_slices)
        row_vmax = max(np.max(z) for z in row_slices)

        last_im = None
        for c, z in enumerate(row_slices):
            ax = fig.add_subplot(gs_2[r, c])
            row_axes_od[r].append(ax)
            last_im = ax.imshow(
                z,
                origin="lower",
                aspect="auto",
                extent=[x.min(), x.max(), y.min(), y.max()],
                vmin=row_vmin,
                vmax=row_vmax,
            )

            if r == 0:
                ax.set_title(f"T={t_targets[c]:.2f}", fontsize=fontsize)
            if r == 2:
                ax.set_xlabel("x", fontsize=fontsize)
            else:
                ax.tick_params(axis="x", labelbottom=False)

            if c == 0:
                ax.set_ylabel("y", fontsize=fontsize)
                ax.tick_params(axis="y", left=True, labelleft=True)
                ax.text(
                    -0.55,
                    0.5,
                    format_param_title(params_od[b]),
                    transform=ax.transAxes,
                    rotation=90,
                    va="center",
                    ha="right",
                    fontsize=fontsize,
                )
            else:
                ax.tick_params(axis="y", left=True, labelleft=False)

            ax.tick_params(axis="both", labelsize=fontsize)

        cbar = fig.colorbar(last_im, ax=row_axes_od[r], fraction=0.02, pad=0.01)
        cbar.ax.tick_params(labelsize=fontsize)

    fig.text(0.23, 0.97, "In-domain", ha="center", fontsize=fontsize + 1, transform=fig.transFigure)
    fig.text(0.755, 0.97, "Out-of-domain", ha="center", fontsize=fontsize + 1, transform=fig.transFigure)

    return fig

In [ ]:
equation = "ReactionDiffusion_2D"
data_folder = f"data/processed/{equation}"
data_train, data_test_id, data_test_od = load_data(data_folder)

fig = create_figure_2d(data_test_id, data_test_od, id_index=[1, 0, 7], od_index=[2,5, 14], fontsize=9, parameter_name =[r"$k$"])
fig.savefig(f"outputs/data_visualisation_{equation}.pdf", bbox_inches="tight")

In [ ]:
# Merge all data visualisation PDFs into one stacked vector PDF with subtitles and separators
import sys

try:
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation

output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)

pdf_specs = [
    ("data_visualisation_Advection.pdf", "A. 1D advection equation"),
    ("data_visualisation_Burgers.pdf", "B. 1D Burgers equation"),
    ("data_visualisation_ReactionDiffusion_1D.pdf", "C. 1D reaction-diffusion equation"),
    ("data_visualisation_ReactionDiffusion_2D.pdf", "D. 2D reaction-diffusion equation"),
]

pdf_paths = [output_dir / name for name, _ in pdf_specs]
missing = [str(p) for p in pdf_paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing one or more per-equation data visualisation PDFs. "
        "Run the equation-specific plotting cells first. Missing: " + ", ".join(missing)
    )

pages = [PdfReader(str(p)).pages[0] for p in pdf_paths]
widths = [float(p.mediabox.width) for p in pages]
heights = [float(p.mediabox.height) for p in pages]

max_width = max(widths)
gap = 15.0
title_band = 15.0
total_height = sum(heights) + gap * (len(pages) - 1) + title_band * len(pages)

merged_page = PageObject.create_blank_page(width=max_width, height=total_height)
cursor_y = total_height
section_title_lines = []
section_bottoms = []

for page in pages:
    w = float(page.mediabox.width)
    h = float(page.mediabox.height)
    x = (max_width - w) / 2.0

    title_top = cursor_y
    title_bottom = title_top - title_band
    y = title_bottom - h

    merged_page.merge_transformed_page(
        page,
        Transformation().translate(tx=x, ty=y),
    )

    section_title_lines.append((title_top + title_bottom) / 2.0)
    section_bottoms.append(y)
    cursor_y = y - gap

# Transparent overlay page: subtitles + gray separators
overlay_fig = plt.figure(figsize=(max_width / 72.0, total_height / 72.0), dpi=72, facecolor="none")
overlay_fig.patch.set_alpha(0)
overlay_ax = overlay_fig.add_axes([0, 0, 1, 1])
overlay_ax.set_xlim(0, 1)
overlay_ax.set_ylim(0, 1)
overlay_ax.axis("off")
overlay_ax.set_facecolor("none")
overlay_ax.patch.set_alpha(0)

for i, (_, subtitle) in enumerate(pdf_specs):
    y_title_n = section_title_lines[i] / total_height
    overlay_ax.text(
        0.025,
        y_title_n,
        subtitle,
        ha="left",
        va="center",
        fontsize=12,
        color="black",
    )
    if i < len(pdf_specs) - 1:
        y_sep_n = (section_bottoms[i] - gap / 2.0) / total_height
        overlay_ax.plot([0.02, 0.98], [y_sep_n, y_sep_n], color="0.7", linewidth=1.0)

overlay_pdf = output_dir / "_data_vis_overlay_tmp.pdf"
overlay_fig.savefig(
    overlay_pdf,
    format="pdf",
    bbox_inches=None,
    pad_inches=0,
    transparent=True,
    facecolor="none",
    edgecolor="none",
)
plt.close(overlay_fig)

overlay_page = PdfReader(str(overlay_pdf)).pages[0]
merged_page.merge_transformed_page(overlay_page, Transformation())

writer = PdfWriter()
writer.add_page(merged_page)
merged_path = output_dir / "data_visualisation_stacked_merged.pdf"
with open(merged_path, "wb") as f:
    writer.write(f)

try:
    overlay_pdf.unlink()
except OSError:
    pass

print(f"Saved merged vector PDF to {merged_path}")